# 06. Проверка чувствительности классификации на окне 2021-01-01 - 2021-01-25

Ноутбук проверяет устойчивость текущей классификации типов слушателей, основанной на правилах,
если пересчитать пользовательские признаки и итоговые типы не на всём январе 2021 года,
а только на периоде с `2021-01-01` по `2021-01-25` включительно.

Сравнение выполняется с текущими итоговыми файлами:
- `data/processed/user_features.csv`;
- `data/processed/user_activity_types.csv`.

В этой проверке устойчивость понимается ограниченно: оценивается, сохраняются ли основные типы слушателей в рамках текущей схемы, основанной на правилах, после пересчёта признаков и самих порогов на сокращённом окне. Изменения дополнительных временных профилей и отдельных признаков интерпретируются отдельно.

Логика перерасчёта согласована с ноутбуками:
- `02_activity_datasets.ipynb`;
- `03_user_features.ipynb`;
- `04_listener_types.ipynb`.

## 1. Импорт библиотек и пути проекта

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TIME_FEATURES_PATH = INTERIM_DIR / "lastfm_time_features.csv"
FULL_USER_FEATURES_PATH = PROCESSED_DIR / "user_features.csv"
FULL_USER_ACTIVITY_TYPES_PATH = PROCESSED_DIR / "user_activity_types.csv"

WINDOW_START = pd.Timestamp("2021-01-01")
WINDOW_END = pd.Timestamp("2021-01-25")

TIME_FEATURES_PATH, FULL_USER_ACTIVITY_TYPES_PATH

(PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/interim/lastfm_time_features.csv'),
 PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/user_activity_types.csv'))

## 2. Константы и функции перерасчёта

In [2]:
MONTH_NUMBER_TO_RU = dict(
    enumerate([
        "Январь",
        "Февраль",
        "Март",
        "Апрель",
        "Май",
        "Июнь",
        "Июль",
        "Август",
        "Сентябрь",
        "Октябрь",
        "Ноябрь",
        "Декабрь",
    ], 1)
)

WEEKDAY_NUMBER_TO_RU = dict(
    enumerate([
        "Понедельник",
        "Вторник",
        "Среда",
        "Четверг",
        "Пятница",
        "Суббота",
        "Воскресенье",
    ])
)

WEEKEND_TO_RU = {
    False: "Будни",
    True:  "Выходные",
}

DAY_PARTS = ["Утро", "День", "Вечер", "Ночь"]

USER_FEATURE_FLOAT_COLUMNS = [
    "active_days_share",
    "morning_share",
    "day_share",
    "evening_share",
    "night_share",
    "weekday_share",
    "weekend_share",
    "avg_daily_plays",
    "median_daily_plays",
    "min_daily_plays",
    "max_daily_plays",
    "std_daily_plays",
    "avg_session_length",
    "median_session_length",
    "max_session_length",
    "avg_plays_per_session",
    "max_plays_per_session",
]

ACTIVITY_TYPES_FLOAT_COLUMNS = [
    "active_days_share",
    "avg_daily_plays",
    "max_daily_plays",
    "evening_share",
    "night_share",
    "avg_session_length",
    "weekend_share",
]

TYPE_DESCRIPTIONS = {
    "Интенсивный": (
        "Пользователь с высоким средним числом прослушиваний в активный день."
    ),
    "Регулярный": (
        "Пользователь со стабильной активностью без экстремально высокой интенсивности."
    ),
    "Эпизодический": (
        "Пользователь с низкой долей активных дней и нерегулярным прослушиванием."
    ),
}

DAY_PART_PROFILE_MAP = {
    "Ночь":  "Ночная активность",
    "Утро":  "Утренняя активность",
    "День":  "Дневная активность",
    "Вечер": "Вечерняя активность",
}

MIXED_ACTIVITY_GAP_THRESHOLD = 0.10
SESSION_GAP_MINUTES = 30


def get_day_part(hour: int) -> str:
    if 6 <= hour <= 11:
        return "Утро"
    if 12 <= hour <= 17:
        return "День"
    if 18 <= hour <= 23:
        return "Вечер"
    return "Ночь"


def get_main_day_part(user_df: pd.DataFrame) -> str:
    counts = user_df["day_part"].value_counts().to_dict()
    return sorted(
        DAY_PARTS,
        key=lambda part: (-counts.get(part, 0), DAY_PARTS.index(part)),
    )[0]


def get_peak_hour(user_df: pd.DataFrame) -> int:
    hour_counts = user_df.groupby("hour").size().reset_index(name="plays")
    hour_counts = hour_counts.sort_values(
        ["plays", "hour"],
        ascending=[False, True],
        kind="stable",
    )
    return int(hour_counts.iloc[0]["hour"])


def get_longest_streak(values: list[bool], expected_value: bool) -> int:
    longest = 0
    current = 0

    for value in values:
        if bool(value) is expected_value:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest


def build_user_daily_activity(df: pd.DataFrame) -> pd.DataFrame:
    users = sorted(df["username"].unique())
    all_dates = pd.date_range(df["date_only"].min(), df["date_only"].max(), freq="D")

    grid = pd.MultiIndex.from_product(
        [users, all_dates],
        names=["username", "date_only"],
    ).to_frame(index=False)

    aggregated = (
        df.groupby(["username", "date_only"], sort=True)
        .agg(
            plays_count=("datetime", "size"),
            first_play_time=("datetime", "min"),
            last_play_time=("datetime", "max"),
            active_hours_count=("hour", "nunique"),
            morning_plays=("day_part", lambda s: int((s == "Утро").sum())),
            day_plays=("day_part", lambda s: int((s == "День").sum())),
            evening_plays=("day_part", lambda s: int((s == "Вечер").sum())),
            night_plays=("day_part", lambda s: int((s == "Ночь").sum())),
        )
        .reset_index()
    )

    result = grid.merge(aggregated, on=["username", "date_only"], how="left")
    result["year"] = result["date_only"].dt.year.astype(int)
    result["month_number"] = result["date_only"].dt.month.astype(int)
    result["weekday_number"] = result["date_only"].dt.weekday.astype(int)
    result["is_weekend"] = result["weekday_number"].isin([5, 6])
    result["month"] = result["month_number"].map(MONTH_NUMBER_TO_RU)
    result["weekday"] = result["weekday_number"].map(WEEKDAY_NUMBER_TO_RU)
    result["weekend"] = result["is_weekend"].map(WEEKEND_TO_RU)

    int_columns = [
        "plays_count",
        "active_hours_count",
        "morning_plays",
        "day_plays",
        "evening_plays",
        "night_plays",
    ]
    result[int_columns] = result[int_columns].fillna(0).astype(int)
    result["has_activity"] = result["plays_count"].gt(0)
    result["first_play_time"] = result["first_play_time"].dt.strftime("%H:%M:%S")
    result["last_play_time"] = result["last_play_time"].dt.strftime("%H:%M:%S")

    ordered_columns = [
        "username",
        "date_only",
        "year",
        "month_number",
        "weekday_number",
        "is_weekend",
        "has_activity",
        "plays_count",
        "first_play_time",
        "last_play_time",
        "active_hours_count",
        "morning_plays",
        "day_plays",
        "evening_plays",
        "night_plays",
        "month",
        "weekday",
        "weekend",
    ]
    return result[ordered_columns].sort_values(["username", "date_only"], kind="stable").reset_index(drop=True)


def build_listening_sessions(df: pd.DataFrame, session_gap_minutes: int = 30) -> pd.DataFrame:
    sorted_df = df.sort_values(["username", "datetime"], kind="stable").copy()
    sorted_df["previous_datetime"] = sorted_df.groupby("username")["datetime"].shift(1)
    sorted_df["time_gap"] = sorted_df["datetime"] - sorted_df["previous_datetime"]
    sorted_df["is_new_session"] = (
        sorted_df["previous_datetime"].isna()
        | sorted_df["time_gap"].gt(pd.Timedelta(minutes=session_gap_minutes))
    )
    sorted_df["session_number"] = (
        sorted_df.groupby("username")["is_new_session"].cumsum().astype(int)
    )

    sessions = (
        sorted_df.groupby(["username", "session_number"], sort=True)
        .agg(
            session_start=("datetime", "min"),
            session_end=("datetime", "max"),
            plays_count=("datetime", "size"),
            active_hours_count=("hour", "nunique"),
        )
        .reset_index()
    )

    sessions = sessions.sort_values(["username", "session_number"], kind="stable").reset_index(drop=True)
    sessions.insert(0, "session_id", range(1, len(sessions) + 1))

    sessions["session_date"] = sessions["session_start"].dt.normalize()
    sessions["year"] = sessions["session_start"].dt.year.astype(int)
    sessions["month_number"] = sessions["session_start"].dt.month.astype(int)
    sessions["weekday_number"] = sessions["session_start"].dt.weekday.astype(int)
    sessions["start_hour"] = sessions["session_start"].dt.hour.astype(int)
    sessions["end_hour"] = sessions["session_end"].dt.hour.astype(int)
    sessions["day_part_start"] = sessions["start_hour"].map(get_day_part)
    sessions["is_weekend"] = sessions["weekday_number"].isin([5, 6])
    sessions["month"] = sessions["month_number"].map(MONTH_NUMBER_TO_RU)
    sessions["weekday"] = sessions["weekday_number"].map(WEEKDAY_NUMBER_TO_RU)
    sessions["weekend"] = sessions["is_weekend"].map(WEEKEND_TO_RU)
    sessions["duration_minutes"] = (
        (sessions["session_end"] - sessions["session_start"])
        .dt.total_seconds()
        .div(60)
        .round()
        .astype(int)
    )

    ordered_columns = [
        "session_id",
        "session_number",
        "username",
        "session_start",
        "session_end",
        "session_date",
        "year",
        "month_number",
        "weekday_number",
        "start_hour",
        "end_hour",
        "day_part_start",
        "is_weekend",
        "plays_count",
        "duration_minutes",
        "active_hours_count",
        "month",
        "weekday",
        "weekend",
    ]
    return sessions[ordered_columns]


def build_user_features(
    listenings_df: pd.DataFrame,
    user_daily_activity_df: pd.DataFrame,
    listening_sessions_df: pd.DataFrame,
) -> pd.DataFrame:
    listenings_df = listenings_df.copy()
    user_daily_activity_df = user_daily_activity_df.copy()
    listening_sessions_df = listening_sessions_df.copy()

    feature_rows = []

    for username in sorted(listenings_df["username"].unique()):
        user_listenings = listenings_df.loc[listenings_df["username"] == username].copy()
        user_days = (
            user_daily_activity_df.loc[user_daily_activity_df["username"] == username]
            .sort_values("date_only", kind="stable")
            .copy()
        )
        user_sessions = listening_sessions_df.loc[listening_sessions_df["username"] == username].copy()

        total_plays       = int(len(user_listenings))
        active_days       = int(user_days["has_activity"].sum())
        inactive_days     = int(len(user_days) - active_days)
        active_days_share = active_days / len(user_days) if len(user_days) > 0 else 0.0

        active_day_counts = user_days.loc[user_days["has_activity"], "plays_count"]
        daily_counts = user_days["plays_count"]

        avg_daily_plays    = float(active_day_counts.mean()) if not active_day_counts.empty else 0.0
        median_daily_plays = int(daily_counts.median()) if not daily_counts.empty else 0
        min_daily_plays    = int(daily_counts.min()) if not daily_counts.empty else 0
        max_daily_plays    = int(daily_counts.max()) if not daily_counts.empty else 0
        std_daily_plays    = float(daily_counts.std(ddof=0)) if not daily_counts.empty else 0.0

        morning_share = float((user_listenings["day_part"] == "Утро").mean())
        day_share     = float((user_listenings["day_part"] == "День").mean())
        evening_share = float((user_listenings["day_part"] == "Вечер").mean())
        night_share   = float((user_listenings["day_part"] == "Ночь").mean())
        weekday_share = float((~user_listenings["is_weekend"]).mean())
        weekend_share = float(user_listenings["is_weekend"].mean())

        main_day_part = get_main_day_part(user_listenings)
        peak_hour = get_peak_hour(user_listenings)
        active_hours_count = int(user_listenings["hour"].nunique())

        session_count         = int(len(user_sessions))
        avg_session_length    = float(user_sessions["duration_minutes"].mean()) if session_count > 0 else 0.0
        median_session_length = int(user_sessions["duration_minutes"].median()) if session_count > 0 else 0
        max_session_length    = int(user_sessions["duration_minutes"].max()) if session_count > 0 else 0
        avg_plays_per_session = float(user_sessions["plays_count"].mean()) if session_count > 0 else 0.0
        max_plays_per_session = int(user_sessions["plays_count"].max()) if session_count > 0 else 0

        activity_flags = user_days["has_activity"].tolist()
        longest_active_streak = get_longest_streak(activity_flags, True)
        longest_inactive_streak = get_longest_streak(activity_flags, False)

        feature_rows.append(
            {
                "username":                username,
                "total_plays":             total_plays,
                "active_days":             active_days,
                "inactive_days":           inactive_days,
                "active_days_share":       active_days_share,
                "avg_daily_plays":         avg_daily_plays,
                "median_daily_plays":      median_daily_plays,
                "min_daily_plays":         min_daily_plays,
                "max_daily_plays":         max_daily_plays,
                "std_daily_plays":         std_daily_plays,
                "morning_share":           morning_share,
                "day_share":               day_share,
                "evening_share":           evening_share,
                "night_share":             night_share,
                "weekday_share":           weekday_share,
                "weekend_share":           weekend_share,
                "main_day_part":           main_day_part,
                "peak_hour":               peak_hour,
                "active_hours_count":      active_hours_count,
                "session_count":           session_count,
                "avg_session_length":      avg_session_length,
                "median_session_length":   median_session_length,
                "max_session_length":      max_session_length,
                "avg_plays_per_session":   avg_plays_per_session,
                "max_plays_per_session":   max_plays_per_session,
                "longest_active_streak":   longest_active_streak,
                "longest_inactive_streak": longest_inactive_streak,
            }
        )

    result = pd.DataFrame(feature_rows).sort_values("username", kind="stable").reset_index(drop=True)
    result[USER_FEATURE_FLOAT_COLUMNS] = result[USER_FEATURE_FLOAT_COLUMNS].round(6)
    return result


def get_threshold(series: pd.Series, quantile: float) -> float:
    return float(series.quantile(quantile))


def build_behavior_profile(row: pd.Series) -> tuple[str, str]:
    shares = [
        (share, float(row[share])) for share in (
            "morning_share",
            "day_share",
            "evening_share",
            "night_share",
        )
    ]
    shares.sort(key=lambda item: item[1], reverse=True)

    top_share = {"name": shares[0][0], "value": shares[0][1]}
    second_share = {"name": shares[1][0], "value": shares[1][1]}
    share_gap = top_share["value"] - second_share["value"]

    if share_gap < MIXED_ACTIVITY_GAP_THRESHOLD:
        return (
            "Смешанная активность",
            (
                f"share_gap < {MIXED_ACTIVITY_GAP_THRESHOLD:.2f}; "
                f"top_share={top_share['name']}; second_share={second_share['name']}"
            ),
        )

    main_day_part = DAY_PART_PROFILE_MAP.get(row["main_day_part"])
    assert main_day_part is not None, (
        f"Unsupported main_day_part value: {row['main_day_part']}"
    )

    return (
        main_day_part,
        (
            f"share_gap >= {MIXED_ACTIVITY_GAP_THRESHOLD:.2f}; "
            f"main_day_part={row['main_day_part']}"
        ),
    )


def assign_listener_types(user_features_df: pd.DataFrame) -> pd.DataFrame:
    user_features_df = user_features_df.copy().sort_values("username", kind="stable").reset_index(drop=True)

    high_intensity_threshold = get_threshold(user_features_df["avg_daily_plays"], 0.75)
    low_activity_threshold = get_threshold(user_features_df["active_days_share"], 0.25)

    rows = []
    for _, row in user_features_df.iterrows():
        if row["avg_daily_plays"] >= high_intensity_threshold:
            listener_type = "Интенсивный"
            assignment_rule = "avg_daily_plays >= upper_quartile"
        elif row["active_days_share"] <= low_activity_threshold:
            listener_type = "Эпизодический"
            assignment_rule = "active_days_share <= lower_quartile"
        else:
            listener_type = "Регулярный"
            assignment_rule = "default regular listener rule"

        behavior_profile, behavior_rule = build_behavior_profile(row)

        rows.append(
            {
                "username":                  row["username"],
                "listener_type":             listener_type,
                "listener_type_description": TYPE_DESCRIPTIONS[listener_type],
                "behavior_profile":          behavior_profile,
                "assignment_rule":           assignment_rule,
                "behavior_rule":             behavior_rule,
                "total_plays":               int(row["total_plays"]),
                "active_days":               int(row["active_days"]),
                "active_days_share":         row["active_days_share"],
                "avg_daily_plays":           row["avg_daily_plays"],
                "max_daily_plays":           int(row["max_daily_plays"]),
                "evening_share":             row["evening_share"],
                "night_share":               row["night_share"],
                "weekend_share":             row["weekend_share"],
                "main_day_part":             row["main_day_part"],
                "peak_hour":                 int(row["peak_hour"]),
                "session_count":             int(row["session_count"]),
                "avg_session_length":        row["avg_session_length"],
            }
        )

    result = pd.DataFrame(rows).sort_values("username", kind="stable").reset_index(drop=True)
    result[ACTIVITY_TYPES_FLOAT_COLUMNS] = result[ACTIVITY_TYPES_FLOAT_COLUMNS].round(6)
    return result

## 3. Загрузка полной выборки и сокращённого временного окна

In [3]:
time_features_df = pd.read_csv(TIME_FEATURES_PATH)
time_features_df["datetime"]   = pd.to_datetime(time_features_df["datetime"])
time_features_df["date_only"]  = pd.to_datetime(time_features_df["date_only"])
time_features_df["is_weekend"] = time_features_df["is_weekend"].astype(bool)

full_user_features_df = pd.read_csv(FULL_USER_FEATURES_PATH)
full_user_activity_types_df = pd.read_csv(FULL_USER_ACTIVITY_TYPES_PATH)

window_df = time_features_df.loc[
    (time_features_df["date_only"] >= WINDOW_START)
    & (time_features_df["date_only"] <= WINDOW_END)
].copy()

print(f"Full period:   {time_features_df['date_only'].min().date()} - {time_features_df['date_only'].max().date()}")
print(f"Window period: {WINDOW_START.date()} - {WINDOW_END.date()}")
print(f"Full shape:    {time_features_df.shape}")
print(f"Window shape:  {window_df.shape}")

Full period:   2021-01-01 - 2021-01-31
Window period: 2021-01-01 - 2021-01-25
Full shape:    (166153, 15)
Window shape:  (23119, 15)


## 4. Пересчёт таблиц активности, признаков и типов слушателей

In [4]:
window_user_daily_activity_df = build_user_daily_activity(window_df)
window_listening_sessions_df = build_listening_sessions(
    window_df,
    session_gap_minutes=SESSION_GAP_MINUTES,
)
window_user_features_df = build_user_features(
    listenings_df=window_df,
    user_daily_activity_df=window_user_daily_activity_df,
    listening_sessions_df=window_listening_sessions_df,
)
window_user_activity_types_df = assign_listener_types(window_user_features_df)

print(f"window_user_daily_activity_df: {window_user_daily_activity_df.shape}")
print(f"window_listening_sessions_df:  {window_listening_sessions_df.shape}")
print(f"window_user_features_df:       {window_user_features_df.shape}")
print(f"window_user_activity_types_df: {window_user_activity_types_df.shape}")

window_user_daily_activity_df: (275, 18)
window_listening_sessions_df:  (1279, 19)
window_user_features_df:       (11, 27)
window_user_activity_types_df: (11, 18)


## 5. Сопоставление итоговых типов и временных профилей

In [5]:
listener_type_comparison_df = (
    full_user_activity_types_df[["username", "listener_type", "behavior_profile"]]
    .merge(
        window_user_activity_types_df[["username", "listener_type", "behavior_profile"]],
        on="username",
        suffixes=("_full", "_window"),
    )
    .sort_values("username", kind="stable")
    .reset_index(drop=True)
)

listener_type_comparison_df["listener_type_changed"] = (
    listener_type_comparison_df["listener_type_full"]
    != listener_type_comparison_df["listener_type_window"]
)

listener_type_comparison_df["behavior_profile_changed"] = (
    listener_type_comparison_df["behavior_profile_full"]
    != listener_type_comparison_df["behavior_profile_window"]
)

listener_type_comparison_df

,username,listener_type_full,behavior_profile_full,listener_type_window,behavior_profile_window,listener_type_changed,behavior_profile_changed
0,Babs_05,Интенсивный,Смешанная активность,Интенсивный,Смешанная активность,False,False
1,Knapster01,Интенсивный,Смешанная активность,Интенсивный,Смешанная активность,False,False
2,Orlenay,Регулярный,Смешанная активность,Регулярный,Смешанная активность,False,False
3,eartle,Регулярный,Смешанная активность,Регулярный,Смешанная активность,False,False
4,franhale,Интенсивный,Смешанная активность,Интенсивный,Дневная активность,False,True
5,isaac,Регулярный,Дневная активность,Регулярный,Дневная активность,False,False
6,jajo,Эпизодический,Дневная активность,Эпизодический,Дневная активность,False,False
7,jonocole,Регулярный,Смешанная активность,Регулярный,Смешанная активность,False,False
8,lobsterclaw,Эпизодический,Смешанная активность,Эпизодический,Дневная активность,False,True
9,massdosage,Регулярный,Смешанная активность,Регулярный,Смешанная активность,False,False


## 6. Сопоставление ключевых пользовательских признаков

In [6]:
feature_comparison_df = (
    full_user_features_df[["username", "total_plays", "avg_daily_plays", "active_days_share"]]
    .merge(
        window_user_features_df[["username", "total_plays", "avg_daily_plays", "active_days_share"]],
        on="username",
        suffixes=("_full", "_to_2021_01_25"),
    )
    .sort_values("username", kind="stable")
    .reset_index(drop=True)
)

feature_comparison_df

,username,total_plays_full,avg_daily_plays_full,active_days_share_full,total_plays_to_2021_01_25,avg_daily_plays_to_2021_01_25,active_days_share_to_2021_01_25
0,Babs_05,33695,1086.935484,1.000000,3730,149.200000,1.00
1,Knapster01,27015,900.500000,0.967742,3567,148.625000,0.96
2,Orlenay,10123,326.548387,1.000000,1899,75.960000,1.00
3,eartle,20966,676.322581,1.000000,3355,134.200000,1.00
4,franhale,32712,1055.225806,1.000000,3990,159.600000,1.00
5,isaac,1780,71.200000,0.806452,386,20.315789,0.76
6,jajo,1102,64.823529,0.548387,159,14.454545,0.44
7,jonocole,17230,555.806452,1.000000,3024,120.960000,1.00
8,lobsterclaw,1063,62.529412,0.548387,170,15.454545,0.44
9,massdosage,19015,613.387097,1.000000,2680,107.200000,1.00


## 7. Итоговая сводная таблица и интерпретация

In [7]:
summary_df = pd.DataFrame(
    {
        "metric": [
            "window_start",
            "window_end",
            "full_total_plays",
            "window_total_plays",
            "listener_type_changes",
            "behavior_profile_changes",
        ],
        "value": [
            WINDOW_START.date().isoformat(),
            WINDOW_END.date().isoformat(),
            int(full_user_features_df["total_plays"].sum()),
            int(window_user_features_df["total_plays"].sum()),
            int(listener_type_comparison_df["listener_type_changed"].sum()),
            int(listener_type_comparison_df["behavior_profile_changed"].sum()),
        ],
    }
)

summary_df

,metric,value
0,window_start,2021-01-01
1,window_end,2021-01-25
2,full_total_plays,166153
3,window_total_plays,23119
4,listener_type_changes,0
5,behavior_profile_changes,3


Если `listener_type_changes = 0`, это означает, что сокращение окна наблюдения до периода `2021-01-01` - `2021-01-25` не меняет основные типы пользователей в рамках текущей схемы, основанной на правилах.

Если при этом изменяются отдельные признаки или дополнительные профили, это не отменяет базовую устойчивость основных классов, но показывает, что временные профили и часть производных характеристик чувствительны к сокращению периода наблюдения.

Поэтому итог этой проверки чувствительности следует трактовать как умеренную устойчивость основной типологии, а не как полную неизменность всех элементов классификации.